# Mutable Mapping(260522)
어떤 타입의 키를 넣어도 무조건 문자열(str)로 변환되어 저장되는 커스텀 딕셔너리를 MutableMapping으로 안전하게 구현한 코드

In [1]:
from collections.abc import MutableMapping

class StringKeyDict(MutableMapping):
    def __init__(self, *args, **kwargs):
        # 실제 데이터가 저장될 내부 공간
        self._inner_dict = dict()
        # 부모의 내장 update() 기능을 빌려 초기 데이터 입력 처리
        self.update(dict(*args, **kwargs))

    # [1] 필수 구현: 쓰기 (수정/추가)
    def __setitem__(self, key, value):
        string_key = str(key)
        self._inner_dict[string_key] = value

    # [2] 필수 구현: 삭제
    def __delitem__(self, key):
        string_key = str(key)
        del self._inner_dict[string_key]

    # [3] 필수 구현: 읽기 (조회)
    def __getitem__(self, key):
        string_key = str(key)
        return self._inner_dict[string_key]

    # [4] 필수 구현: 반복
    def __iter__(self):
        return iter(self._inner_dict)

    # [5] 필수 구현: 길이
    def __len__(self):
        return len(self._inner_dict)

    def __repr__(self):
        return f"{self.__class__.__name__}({self._inner_dict!r})"

In [7]:
# 코드 동작 테스트

# 객체 생성 및 데이터 추가
my_dict = StringKeyDict()
my_dict[123] = "숫자 키"       # 정수형 123을 넣었지만 문자열 "123"으로 자동 변환
my_dict["name"] = "Alice"    # 문자열은 그대로 문자열로 입력

print("초기 상태:", my_dict)
# 출력: StringKeyDict({'123': '숫자 키', 'name': 'Alice'})


초기 상태: StringKeyDict({'123': '숫자 키', 'name': 'Alice'})


In [8]:
# 자동으로 제공되는 pop() 메서드 사용
popped = my_dict.pop(123)
print(f"pop 결과: {popped} / 남은 데이터: {my_dict}")
# 출력: pop 결과: 숫자 키 / 남은 데이터: StringKeyDict({'name': 'Alice'})


pop 결과: 숫자 키 / 남은 데이터: StringKeyDict({'name': 'Alice'})


In [ ]:

# 자동으로 제공되는 update() 메서드 사용
my_dict.update({456: "새로운 숫자", "age": 25})   # 숫자 456은 문자열 키로 변환되고, 내용도 갱신됨 (260522)
print("update 후:", my_dict)
# 출력: update 후: StringKeyDict({'name': 'Alice', '456': '새로운 숫자', 'age': 25})


update 후: StringKeyDict({'name': 'Alice', '456': '새로운 숫자', 'age': 25})


In [10]:

# 자동으로 제공되는 items() 루프 출력
for key, value in my_dict.items():
    print(f"키 타입: {type(key)} | 키: {key} | 값: {value}")

    
# 출력:
# 키 타입: <class 'str'> | 키: name | 값: Alice
# 키 타입: <class 'str'> | 키: 456 | 값: 새로운 숫자
# 키 타입: <class 'str'> | 키: age | 값: 25

키 타입: <class 'str'> | 키: name | 값: Alice
키 타입: <class 'str'> | 키: 456 | 값: 새로운 숫자
키 타입: <class 'str'> | 키: age | 값: 25


- 인터페이스 중심 설계: 함수를 설계할 때 무조건 dict를 고집하기보다, 읽기 전용 매핑 객체가 필요하다면 Mapping을, 수정이 필요한 객체라면 MutableMapping을 타입 힌트와 타입 검사에 적극 활용하세요.

- 상속 부작용 방지: 커스텀 딕셔너리를 만들 때 내장 dict 상속은 피하고, MutableMapping을 상속받아 구현하는 것이 견고하고 안전한 파이썬 코드를 작성하는 정석입니다.

## 딕셔너리를 바라보는 시야의 확장: Mapping
- 타입 검사의 한계와 유연성 부족
많은 개발자가 특정 함수가 딕셔너리를 매개변수로 받아야 할 때 아래와 같이 타입을 엄격하게 제한하곤 합니다.
- 하지만 파이썬에는 내장 dict 외에도 collections.defaultdict, collections.OrderedDict, 혹은 사용자가 커스텀하게 정의한 딕셔너리 객체 등 딕셔너리처럼 동작하는 다양한 객체가 존재합니다.

In [4]:
# 아쉬운 방식: 오직 dict 타입만 허용함
def process_data(data):
    if not isinstance(data, dict):
        raise TypeError("데이터는 dict 타입이어야 합니다.")
    return data.get("status")

- Mapping을 활용한 덕 타이핑(Duck Typing) 구현
Mapping을 사용하면 키-값 쌍으로 조회 가능한 모든 딕셔너리 계열의 객체를 아우르는 타입 힌트와 검사가 가능해집니다.

In [5]:
from collections.abc import Mapping

# 올바른 방식: 딕셔너리처럼 동작하는 모든 객체를 허용 (확장성)
def process_data(data: Mapping[str, any]) -> any:
    if not isinstance(data, Mapping):
        raise TypeError("데이터는 Mapping 인터페이스를 구현해야 합니다.")
    return data.get("status")

- isinstance(data, Mapping)은 객체의 실제 타입이 dict인지 검사하는 것이 아니라, 내부적으로 딕셔너리의 핵심 키 조회 메서드(getitem, iter, len)를 가지고 있는지 검사합니다.

## 불변성(Immutability) 지향과 데이터 보호
- dict 타입 힌트를 사용하면 함수 내부에서 데이터를 마음대로 변경(추가, 수정, 삭제)해도 된다는 오해를 살 수 있습니다. 반면 Mapping은 읽기 전용(Read-Only) 인터페이스를 의미합니다.

- dict 힌트: "이 객체는 읽고 쓸 수 있는 딕셔너리야."

- Mapping 힌트: "이 객체는 키로 값을 조회할 수만 있어. 수정은 하지 마."

In [6]:
from collections.abc import Mapping

# 이 함수는 데이터를 읽기만 하겠다는 의도를 명확히 전달합니다.
def render_profile(config: Mapping[str, str]):
    # 읽기는 정상 작동
    print(f"User: {config['username']}")
    
    # ❌ 정적 타입 체커(mypy 등)가 경고를 발생시킴 (Mapping에는 __setitem__이 없음)
    # config["last_login"] = "2026-05-22"

## MutableMapping 구현의 5대 필수 메서드
- 개발을 하다 보면 "키를 조회할 때마다 로그를 남기는 딕셔너리"나 "데이터를 문자열로 강제 변환하는 딕셔너리" 같은 커스텀 객체를 만들어야 할 때가 있습니다.

- 이때 내장 dict 클래스를 직접 상속받으면 안 됩니다. 내장 dict는 C 언어 수준의 속도 최적화를 위해 일부 메서드가 사용자가 오버라이드한 메서드를 무시하는 부작용(예: dict.update()가 __setitem__을 거치지 않는 현상)이 있습니다.

- 수정, 삭제가 모두 가능한 안전한 커스텀 딕셔너리를 만들려면 MutableMapping을 상속받아야 합니다.

| 분류 | 매직 메서드 (Magic Method) | 파이썬 내장 동작 연동 | 역할 및 구현 내용 |
| :--- | :--- | :--- | :--- |
| **읽기 (Read)** | `__getitem__(self, key)` | `value = obj[key]` | 특정 키에 해당하는 값을 조회하고 반환합니다. |
| | `__iter__(self)` | `for key in obj:`<br>`keys()` | 딕셔너리의 키들을 순회할 수 있는 이터레이터를 반환합니다. |
| | `__len__(self)` | `len(obj)` | 딕셔너리에 저장된 총 아이템(키-값 쌍)의 개수를 반환합니다. |
| **쓰기 (Write)** | `__setitem__(self, key, value)` | `obj[key] = value` | 새로운 키-값 쌍을 추가하거나, 기존 키의 값을 수정합니다. |
| | `__delitem__(self, key)` | `del obj[key]` | 특정 키와 그에 해당하는 값을 딕셔너리에서 삭제합니다. |

### 덕 타이핑 (Duck Typing)


In [11]:
class Duck:
    def quack(self):
        print("깩깩!")

class Person:
    def quack(self):
        print("인간이 오리 소리를 냅니다: 깩깩!")

# 객체의 타입을 검사하지 않고, 오직 'quack' 메서드가 있는지만 보고 실행합니다.
def make_it_quack(animal):
    animal.quack()

duck = Duck()
person = Person()

make_it_quack(duck)    # 출력: 깩깩!
make_it_quack(person)  # 출력: 인간이 오리 소리를 냅니다: 깩깩!

깩깩!
인간이 오리 소리를 냅니다: 깩깩!


## 덕 타이핑 : 실무 사례

In [16]:
class TossPay:
    def process_toss(self, amount):
        print(f"토스페이로 {amount}원 결제 완료")

class KakaoPay:
    def process_kakao(self, amount):
        print(f"카카오페이로 {amount}원 결제 완료")

# [문제의 결제 처리 함수] 새로운 결제사가 추가될 때마다 함수 내부가 엉망이 됩니다.
def pay_service(payment_method, amount):
    if isinstance(payment_method, TossPay):
        payment_method.process_toss(amount)
    elif isinstance(payment_method, KakaoPay):
        payment_method.process_kakao(amount)
    else:
        raise TypeError("지원하지 않는 결제 수단입니다.")

In [17]:
pay_service(NaverPay(), 50000)

TypeError: 지원하지 않는 결제 수단입니다.

In [ ]:
# 각 클래스에 동일한 이름의 pay() 메서드를 정의

class TossPay:
    def pay(self, amount):
        print(f"토스페이로 {amount}원 결제")

class KakaoPay:
    def pay(self, amount):
        print(f"카카오페이로 {amount}원 결제")

class Payco:
    def pay(self, amount):
        print(f"페이코로 {amount}원 결제")

# [덕 타이핑을 적용한 결제 처리 함수]
def pay_service(payment_method, amount):             # 각 클래스에 동일한 이름으로 정의된 pay() 메서드를 호출함 : 확장성이 용이함
    # payment_method가 어떤 클래스인지 검사하지 않습니다.
    # "오리처럼 결제(pay)할 수 있으면 오리(결제 수단)다!"
    payment_method.pay(amount)

In [19]:
# 새로 추가된 네이버페이
class NaverPay:
    def pay(self, amount):
        print(f"네이버페이로 {amount}원 결제")

# 기존 결제 함수를 그대로 사용 가능! (확장성 폭발)
pay_service(NaverPay(), 50000)
# 출력: 네이버페이로 50000원 결제

네이버페이로 50000원 결제
